# Homework: Agentic RAG

In this homework, we build a RAG system from scratch and then make it
agentic - the same path as the module.

Instead of the course FAQ, our knowledge base is the course lessons
themselves.


## Preparation

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

## Question #1. How many lesson pages

How many lesson pages are in the dataset?

* 24
* 72
* 240
* 720

In [5]:
print(f"The number of files is: {len(files)}.")

The number of files is: 72.


> **Answer**: The number of files is: 72.


## Question #2. Indexing and searching

Index the documents with minsearch - make `content` a text field and
`filename` a keyword field. Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

* `01-agentic-rag/lessons/03-rag.md`
* `01-agentic-rag/lessons/14-agentic-loop.md`
* `04-evaluation/lessons/13-llm-as-judge.md`
* `06-best-practices/lessons/02-hybrid-search.md`

In [6]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [7]:
query = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    query=query,
    filter_dict={},
    boost_dict={"content": 1.0},
    num_results=5,
)

In [8]:
print(f"First result is: {search_results[0]["filename"]}")

First result is: 01-agentic-rag/lessons/14-agentic-loop.md


> **Answer**: `01-agentic-rag/lessons/14-agentic-loop.md`

## Question #3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper 
script we prepared during the lessons:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
```

`RAGBase` was written for the FAQ schema (`section`/`question`/`answer`),
while our documents have `filename` and `content`.

Two solutions are possible:

- Implement the RAG flow yourself
- Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

> How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for
this request?

* 700
* 7000
* 70000
* 700000

We count input tokens instead of price because the cost depends on the model
and provider you use, but the size of the prompt we send is the same for
everyone.

Most LLM APIs report token usage on the response object (e.g.
`response.usage.input_tokens` / `prompt_tokens`). We'll read the input tokens
from there.

You will need to modify the code for the rag helper to expose the usage.

In the RAG Helper class, `llm` returns only the text. Modify it to return the whole response, and change `rag` to return both the answer and usage (as a tuple or create a small dataclass for that).

Note: for this question and the next ones, if your answer doesn't match exactly,
just select the closest option - especially if you use a different model or a
different LLM provider.


In [9]:
from rag_helper import RAGBase

rag = RAGBase(index, openai_client)
query = "How does the agentic loop keep calling the model until it stops?"
results = rag.rag(query)


Input tokens: 7036
Output tokens: 95


> **Answer:** Input tokens were 7036, approx 7000 tokens.

## Question #4. Chunking

The lesson pages are long - some are thousands of characters. Long documents
make retrieval less precise: a match deep inside a page still pulls in the
whole page. A common fix is chunking: split each page into smaller,
overlapping pieces and index those instead.

gitsource has a helper for this: `chunk_documents`. It uses a sliding
window - a window of `size` characters slides across the text in steps of
`step` characters, and each window position becomes one chunk:

```python
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
```

With `size=2000` and `step=1000` (you can see the implementation
[here](https://github.com/alexeygrigorev/gitsource/blob/master/gitsource/chunking.py)):

- Each chunk is a window of `size` characters of the page.
- The window moves forward by `step` characters between chunks. Since `step`
  is smaller than `size`, consecutive chunks overlap by `size - step` (1000)
  characters, so a passage split across a boundary still appears whole in one
  of the chunks.
- Every chunk keeps the original fields (`filename`) and adds `start` (the
  offset in the page) and `content` (the chunk text).

How many chunks do you get?

* 70
* 295
* 1100
* 45

In [10]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [11]:
print(f"Number of chunks are: {len(chunks)}")

Number of chunks are: 295


> **Answer:** 295 chunks.

## Question #5. RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the
LLM. Let's measure that.

Index the chunks from Q4 (same as before: `content` as a text field,
`filename` as a keyword field), point your RAG at the chunk index, and
answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked
version send?

* about the same
* 3× fewer
* 10× fewer
* 30× fewer


In [12]:
chunked_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunked_index.fit(chunks)

In [13]:
from rag_helper import RAGBase

rag = RAGBase(chunked_index, openai_client)
query = "How does the agentic loop keep calling the model until it stops?"
results = rag.rag(query)


Input tokens: 2221
Output tokens: 117


> **Answer:** Input tokens with chunked index were 2221, approx 3x fewer.

## Question #6. Turning it into an agent

So far search runs once, with the exact query. Let's make it agentic: give
the LLM a `search` tool and let it decide when (and what) to search. We
suggest [toyaikit](https://github.com/alexeygrigorev/toyaikit), the small
agent library from the module, but you can use anything you like - the OpenAI
Agents SDK, PydanticAI, LangChain, or a hand-written loop.

If you go with toyaikit:

```bash
uv add toyaikit
```

Create a `search` function that uses the chunk index. Give it a type hint and
a docstring - most frameworks read them to build the tool schema for you.

Build an agent with your `search` tool and run it (with toyaikit, the same way
as in the ToyAIKit lesson). Use these instructions for the agent (they nudge
it to search a few times):

> You're a course teaching assistant. Answer the student's question using the
> search tool. Make multiple searches with different keywords before answering.

Ask it:

> How does the agentic loop work, and how is it different from plain RAG?

The agent decides on its own when to search and when to answer. Count how many
times it called the `search` tool.

How many times did the agent call `search`?

> Note: the agent decides this itself, so it varies a little between runs -
> pick the closest option. We measured this with OpenAI `gpt-5.4-mini`; with a
> different model or provider the number may differ, so keep that in mind.

* 0
* 4
* 10
* 20


In [14]:
!uv add toyaikit

Resolved 128 packages in 0.86ms
Checked 124 packages in 1ms


In [15]:
def search_tool(keyword: str) -> str:
    results = chunked_index.search(
        query=keyword,
        boost_dict={"content": 1.0},
        num_results=3
    )
    return "\n\n".join([r['content'] for r in results])

system_instructions = (
    "You're a course teaching assistant. Answer the student's question using the search tool. "
    "Make multiple searches with different keywords before answering."
)

user_question = "How does the agentic loop work, and how is it different from plain RAG?"

In [16]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [17]:
agent_tools = Tools()
agent_tools.add_tool(search_tool)
agent_tools.get_tools()


[{'type': 'function',
  'name': 'search_tool',
  'description': 'No description provided.',
  'parameters': {'type': 'object',
   'properties': {'keyword': {'type': 'string',
     'description': 'keyword parameter'}},
   'required': ['keyword'],
   'additionalProperties': False}}]

In [18]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=system_instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)
result = runner.loop(
    prompt=user_question,
    callback=callback,
)

-> Response received


-> Response received


> **Answer:** Agent called `search_tool` three (3) times. Approx. 4 times.